# Runner — Motor GES/CAEC lite

Interactive front end for the engine: run the tests and try the functions by hand.

- **Kernel:** select the repo's `.venv` (Python 3.12). It needs `pip install -r requirements-dev.txt`.
- **No logic lives here.** Every cell calls code in `src/`. If a cell starts computing something itself, that code belongs in `src/` with a test.
- `autoreload` is on, so edits to `src/*.py` show up without restarting the kernel.
- Before committing: clear all outputs (*Clear All Outputs*), so git diffs stay readable.

In [ ]:
%load_ext autoreload
%autoreload 2

import subprocess
import sys
from datetime import date
from pathlib import Path

ROOT = Path.cwd()
assert (ROOT / "src" / "calculo.py").exists(), "Open this notebook from the repo root"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from datetime import datetime

from src import calculo, repositorio


def pytest(*args):
    """Run pytest in a subprocess, so it always sees the current code."""
    r = subprocess.run([sys.executable, "-m", "pytest", *args], cwd=ROOT, capture_output=True, text=True)
    print(r.stdout, r.stderr)


def tabla(filas):
    print(f"{'id':>3}  {'fecha':<10}  {'copago':>11}  {'paga benef.':>11}  {'cubre seguro':>12}  {'acumulado':>11}")
    for f in filas:
        print(f"{f.prestacion_id:>3}  {f.fecha}  {f.copago:>11,}  {f.paga_beneficiario:>11,}  {f.cubre_seguro:>12,}  {f.acumulado:>11,}")

## 1. Tests

The same command CI will run. Pass pytest flags as arguments, e.g. `pytest("-v", "-k", "desglose")`.

In [ ]:
pytest("-q")

In [ ]:
pytest("-v", "-k", "frontera or solapamiento")

## 2. Rules: which version applies on a date (S03)

Same two GES01 versions as `sql/seed.sql`: 122 UF until 2025-12-31, 130 UF from 2026-01-01.

In [ ]:
REGLAS = [
    calculo.ReglaGES(1, "GES01", date(2025, 1, 1), date(2025, 12, 31), 29, 122.0),
    calculo.ReglaGES(2, "GES01", date(2026, 1, 1), None, 29, 130.0),
]

for d in [date(2024, 12, 31), date(2025, 12, 31), date(2026, 1, 1)]:
    r = calculo.regla_vigente("GES01", d, REGLAS)
    print(d, "→", f"regla {r.regla_id}, tope {r.tope_uf} UF" if r else "no rule (gap)")

## 3. Deductible and split for beneficiary 1 (S03)

Beneficiary 1 pays 5.0 UF/month. The window starts 2025-03-10, so rule 1 applies and the UF is the one for 2025-02-28 (plan D3).
Prestaciones are typed in here by hand until the database layer exists (S04).

In [ ]:
regla = calculo.regla_vigente("GES01", date(2025, 3, 10), REGLAS)
ded_uf = calculo.deducible_ges_uf(5.0, regla)
ded_clp = calculo.uf_a_clp(ded_uf, 38_630.75)
print(f"deducible: {ded_uf} UF = {ded_clp:,} CLP")


def p(pid, fecha, copago):
    return calculo.Prestacion(pid, 1, fecha, None, "GES01", True, False, copago * 5, copago)


PRESTACIONES = [
    p(1, date(2025, 3, 10), 1_000_000),
    p(2, date(2025, 5, 20), 1_200_000),
    p(5, date(2025, 8, 5), 1_500_000),
    p(6, date(2025, 10, 15), 800_000),
    p(7, date(2025, 11, 20), 800_000),   # the late arrival
    p(8, date(2026, 2, 10), 600_000),
]
tabla(calculo.desglose(PRESTACIONES, ded_clp))

### What-if: before the late arrival was known

Drop prestación 7 (received 2026-03-05) and the crossing moves from 2025-11-20 to 2026-02-10.

In [ ]:
tabla(calculo.desglose([x for x in PRESTACIONES if x.prestacion_id != 7], ded_clp))

## 4. The database (S04)

Fresh in-memory database with `sql/schema.sql` + `sql/seed.sql`. Nothing is written to disk.
Change `CONOCIDO_HASTA` to see what the system knew at another moment (plan D5): `None` means everything known now.

In [ ]:
conn = repositorio.conectar(":memory:")
repositorio.inicializar(conn)

ben = repositorio.leer_beneficiario(conn, 1)
print(ben)
for r in repositorio.leer_reglas_ges(conn, "GES01"):
    print(r)
print("UF 2025-02-28:", repositorio.leer_uf(conn, date(2025, 2, 28)))

In [ ]:
CONOCIDO_HASTA = None                       # try: datetime(2026, 3, 4, 23, 59)

print(f"{'id':>3}  {'fecha':<10}  {'recibido_en':<19}  {'código':<6}  {'red GES':<7}  {'copago':>11}")
for x in repositorio.leer_prestaciones(conn, 1, CONOCIDO_HASTA):
    print(f"{x.prestacion_id:>3}  {x.fecha_prestacion}  {x.recibido_en}  {x.codigo_ges or '-':<6}  {str(x.en_red_ges):<7}  {x.monto_copago:>11,}")

## Next

S05 adds `motor.calcular`: the filtering, window, rule and deductible steps done by hand in section 3 will run on this data in one call, and the result gets saved.